In [1]:
import time
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,
                                        cross_val_predict, GridSearchCV, RandomizedSearchCV)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                                recall_score, f1_score, roc_auc_score)

In [32]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
print(df.shape)


(7043, 21)


In [33]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["churn"] = (df["Churn"] == "Yes").astype(int)

num = ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]
cat = ["Contract", "InternetService", "PaymentMethod",
        "PaperlessBilling", "Partner", "Dependents", "TechSupport"]

X = pd.get_dummies(df[num + cat], columns=cat, dtype=int)
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X.shape, X_train.shape, X_test.shape)
print("train churners", y_train.sum(), round(y_train.mean(), 4),
        "| test churners", y_test.sum(), round(y_test.mean(), 4))

(7043, 23) (5634, 23) (1409, 23)
train churners 1495 0.2654 | test churners 374 0.2654


- we do not need a separate validation set when we use 5-fold cross validation on X_train because 5-fold cross validation handles the validation process internally by automatically splitting the data.
- to ensure strict reproducibility and a fair, apples-to-apples comparison across models.

## Rules
R1. Compare models with 5-fold StratifiedKFold (shuffle=True,random_state=42) on X_train, scoring = roc_auc.

R2. "Tie band" = one standard error of the best model = std of its 5 fold scores divided by the square root of 5. (One fold is noisy; the AVERAGE of five folds is steadier.)

R3. Inside the tie band, choose the SIMPLER setting: smaller max_depth first, then larger min_samples_leaf.

R4. The forest must beat logistic regression by MORE than the band,otherwise we choose logistic regression (simpler, explainable).

R5. Threshold: among thresholds with recall >= 0.70, take the best F1(same rule as Days 9 and 10), chosen on out-of-fold predictions.

R6. Test set: once, at the end. No decision uses it.


In [36]:
sfk = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv(model):
    s = cross_val_score(model, X_train, y_train, cv=sfk, scoring="roc_auc")
    return round(float(s.mean()), 4), round(float(s.std()), 4)

def make_lr():
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ])

def make_forest(**kw):
    params = dict(n_estimators=300, random_state=42, n_jobs=-1)
    params.update(kw)
    return Pipeline([("imp", SimpleImputer(strategy="median")),
                     ("model", RandomForestClassifier(**params))])


In [38]:
print("baseline      ", cv(Pipeline([("m", DummyClassifier(strategy="most_frequent"))])))
print("logistic     ", cv(make_lr()))
print("default RF   ", cv(make_forest()))
print("Day10 RF 5/5 ", cv(make_forest(max_depth=5, min_samples_leaf=5))) 

baseline       (0.5, 0.0)
logistic      (0.8422, 0.0122)
default RF    (0.8102, 0.0122)
Day10 RF 5/5  (0.8443, 0.0106)


- because day 9 logistic regression was evaluated on a single split, while here it uses 5-fold CV score averages performance across 5 different validation split.
- The 5-fold cross-validation score (0.8422) is far more trustworthy because by a single split is vulnerable to select while here it validates 5 times, and CV eliminates split bias, and gives you more stable and realistics estimate of how the pipeline generalizes to unseen data.
- 0.0106/sqrt(5) = 0.0047, the performance gap is smaller so it is a tie.

In [41]:
grid = {"model__max_depth": [5, 8, 12, None],
        "model__min_samples_leaf": [1, 5, 10, 20]}

t0 = time.time()
gs = GridSearchCV(make_forest(), grid, cv=sfk, scoring="roc_auc")
gs.fit(X_train, y_train)
print("seconds", round(time.time() - t0))

res = pd.DataFrame(gs.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf", "mean_test_score", "std_test_score", "rank_test_score"]]
res.columns = ["max_depth", "min_leaf", "mean", "std", "rank"]
res = res.sort_values("rank")
print(res.round(4).to_string(index=False))

best = res.iloc[0]
band = best["std"] / 5 ** 0.5

tied = res[res["mean"] >= best["mean"] - band].copy()
print("band", round(band, 4), "| tied", len(tied), "of", len(res))
tied["depth_key"] = tied["max_depth"].fillna(999)
pick = tied.sort_values(["depth_key", "min_leaf"], ascending=[True, False])
print("rule pick (R3):", pick["max_depth"], pick["min_leaf"], round(pick["mean"], 4), round(pick["std"], 4))

seconds 59
max_depth  min_leaf   mean    std  rank
        8        20 0.8448 0.0106     1
     None        20 0.8448 0.0108     2
       12        20 0.8447 0.0108     3
        8        10 0.8443 0.0107     4
        5         5 0.8443 0.0106     5
        5        20 0.8442 0.0105     6
        5         1 0.8442 0.0106     7
        5        10 0.8441 0.0105     8
       12        10 0.8440 0.0108     9
     None        10 0.8438 0.0108    10
        8         5 0.8436 0.0104    11
        8         1 0.8421 0.0107    12
       12         5 0.8396 0.0108    13
     None         5 0.8391 0.0114    14
       12         1 0.8267 0.0110    15
     None         1 0.8102 0.0122    16
band 0.0047 | tied 12 of 16
rule pick (R3): 3        5
2        5
1        5
0        5
7        8
6        8
5        8
4        8
11      12
10      12
15    None
14    None
Name: max_depth, dtype: object 3     20
2     10
1      5
0      1
7     20
6     10
5      5
4      1
11    20
10    10
15    20
14 

In [43]:
grid = {"model__max_depth": [5, 8, 12, None],
            "model__min_samples_leaf": [1, 5, 10, 20]}
t0 = time.time()
gs = GridSearchCV(make_forest(), grid, cv=sfk, scoring="roc_auc")
gs.fit(X_train, y_train)
print("seconds", round(time.time() - t0))

res = pd.DataFrame(gs.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf",
                                        "mean_test_score", "std_test_score", "rank_test_score"]]
res.columns = ["max_depth", "min_leaf", "mean", "std", "rank"]
res = res.sort_values("rank")
print(res.round(4).to_string(index=False))

best = res.iloc[0]
band = best["std"] / 5 ** 0.5                 # R2: one standard error
tied = res[res["mean"] >= best["mean"] - band].copy()
print("band", round(band, 4), "| tied", len(tied), "of", len(res))
tied["depth_key"] = tied["max_depth"].fillna(999)      # None counts as the deepest
pick = tied.sort_values(["depth_key", "min_leaf"], ascending=[True, False]).iloc[0]
print("rule pick (R3):", pick["max_depth"], pick["min_leaf"], round(pick["mean"], 4), round(pick["std"], 4))


seconds 60
max_depth  min_leaf   mean    std  rank
        8        20 0.8448 0.0106     1
     None        20 0.8448 0.0108     2
       12        20 0.8447 0.0108     3
        8        10 0.8443 0.0107     4
        5         5 0.8443 0.0106     5
        5        20 0.8442 0.0105     6
        5         1 0.8442 0.0106     7
        5        10 0.8441 0.0105     8
       12        10 0.8440 0.0108     9
     None        10 0.8438 0.0108    10
        8         5 0.8436 0.0104    11
        8         1 0.8421 0.0107    12
       12         5 0.8396 0.0108    13
     None         5 0.8391 0.0114    14
       12         1 0.8267 0.0110    15
     None         1 0.8102 0.0122    16
band 0.0047 | tied 12 of 16
rule pick (R3): 5 20 0.8442 0.0105


- 1st rank, it is 5 rank far from the CV best.
- day 10 5 settings while here 16 settings, this tells you that when settings are statistically tied, the minor shifts in ranking are down to pure randomness. Chasing the absolute "winner" based on decimal places is a trap; it is much wiser to select based on structural simplicity. 
- The 4 combinations that are clearly worse are ranks 13 through 16: 12/5, None/5, 12/1, and None/1. They all combine high or unlimited depth (12 or None) with tiny leaf sizes (1 or 5). This allows individual trees to grow unconstrained and split endlessly, forcing the forest to overfit by memorising individual noisy training rows
- Choosing a smaller, smoother forest is the better pick because it enforces strict regularization to protect against overfitting, runs faster, and uses less memory while delivering the exact same statistical performance as the complex models.

In [44]:
for n in (50, 100, 300, 600):
    print("n_estimators", n, cv(make_forest(max_depth=5, min_samples_leaf=5, n_estimators=n)))

for mf in ("sqrt", 0.5, 1.0):
    print("max_features", mf, cv(make_forest(max_depth=5, min_samples_leaf=5, max_features=mf)))

n_estimators 50 (0.8428, 0.0114)
n_estimators 100 (0.843, 0.0107)
n_estimators 300 (0.8443, 0.0106)
n_estimators 600 (0.8443, 0.0103)
max_features sqrt (0.8443, 0.0106)
max_features 0.5 (0.8451, 0.0106)
max_features 1.0 (0.844, 0.0117)


- going from 300 to 600 the mean scores remain the same, eventhough the trees grows. Adding more trees does not make the model smarter, because at certain threshold more votes make the average steadier until it can’t get steadier, the change from 300 to 600 is 0.0000. More trees cost time but stop helping
- No it is a tie because the improvement is smaller than the band (0.0047). This is just a random noise from cv.
- max_depth and min_leaf moved the score drastically by 0.0346 (from a broken 0.8102 up to a robust 0.8448).
- n_estimators and max_features barely budged the score by 0.0015 or less.

In [47]:
dist = {"model__max_depth": randint(2, 16),
        "model__min_samples_leaf": randint(1, 60),
        "model__max_features": [0.2, 0.3, 0.5, "sqrt"],
        "model__n_estimators": [100, 200, 300]}
t0 = time.time()
rs = RandomizedSearchCV(make_forest(), dist, n_iter=20, cv=sfk, scoring="roc_auc", random_state=42)

rs.fit(X_train, y_train)
print("seconds", round(time.time() - t0))

r2 = pd.DataFrame(rs.cv_results_)[["param_model__max_depth", "param_model__min_samples_leaf",
                                       "param_model__max_features", "param_model__n_estimators",
                                       "mean_test_score", "std_test_score", "rank_test_score"]]
r2.columns = ["depth", "leaf", "max_feat", "n_est", "mean", "std", "rank"]
r2 = r2.sort_values("rank")
print(r2.head(6).round(4).to_string(index=False))
print("worst:", r2.iloc[-1].to_dict())
print("best :", rs.best_params_, round(rs.best_score_, 4))

seconds 23
 depth  leaf max_feat  n_est   mean    std  rank
     9    35      0.5    200 0.8462 0.0112     1
     7    21      0.3    100 0.8458 0.0106     2
    13    44      0.5    100 0.8458 0.0111     3
    12    42      0.5    300 0.8455 0.0119     4
    15    47      0.3    200 0.8453 0.0112     5
    13    22      0.3    100 0.8451 0.0100     6
worst: {'depth': 2, 'leaf': 50, 'max_feat': 'sqrt', 'n_est': 200, 'mean': 0.8371605614135611, 'std': 0.010537429116820238, 'rank': 20}
best : {'model__max_depth': 9, 'model__max_features': 0.5, 'model__min_samples_leaf': 35, 'model__n_estimators': 200} 0.8462


- It is a tie because the difference is just 0.0014. This gap is smaller than 0.0047.
- total fits: 9912 * 5 = 49,560. RandomizedSearchCV samples fixed random settings which is 20 combinitions here, with only 100 total fits •  it maps out the boundaries of the hyperparameter space well enough to land on a near-optimal model configuration within about a minute.
- The rowss have minimum leaf size range from 21 to 47. This forces individual trees to pool large blocks of data together at their base nodes, ensuring the model generalizes smoothly instead of splitting down to noisy, isolated data points.
- It tell about heavy cost of underfitting, because restricting the model to a max depth 2 prevent the trees from executing enough splits to capture the actual patterns of dataset.

In [ ]:
forest5 = lambda: RandomForestClassifier(n_estimators=300, max_depth=5,
                                         min_samples_leaf=5, random_state=42, n_jobs=-1)
scaled = Pipeline([("imp", SimpleImputer(strategy="median")),
                   ("sc", StandardScaler()), ("model", forest5())])
print("forest scaled", cv(scaled))
for k in (8, 12, 16, 23):
    p = Pipeline([("imp", SimpleImputer(strategy="median")),
                  ("sel", SelectKBest(f_classif, k=k)), ("model", forest5())])
    print("k =", k, cv(p))

- scaled and unscaled gives the same mean because tree-based models look at features independently and only ask if a value is above or below a threshold, meaning shifting or stretching the scale of a column does not alter where those mathematical boundaries are split.
- No, it is a statistical tie, and SelectKBest should be excluded from the final model because the score bump of 0.0014 (0.8457 - 0.8443) falls completely inside the standard error tie band of 0.0047, meaning the increase is just random cross-validation fold noise. Under the simplicity rule, when scores tie, you choose the cleaner architecture. Adding a feature selection step introduces unnecessary pipeline complexity without delivering a genuine performance victory.
- SelectKBest must sit inside the pipeline to prevent data leakage during cross-validation by ensuring features are selected using only the active training folds.

In [ ]:
finalist = make_forest(max_depth=5, min_samples_leaf=20)     # the R3 pick from Part 3
f_scores = cross_val_score(finalist, X_train, y_train, cv=sfk, scoring="roc_auc")
l_scores = cross_val_score(make_lr(), X_train, y_train, cv=sfk, scoring="roc_auc")
print("forest", f_scores.mean().round(4), f_scores.std().round(4),
      "| LR", l_scores.mean().round(4), l_scores.std().round(4))
print("fold differences forest - LR:", (f_scores - l_scores).round(4),
      "| forest wins", int((f_scores > l_scores).sum()), "of 5")
print("mean difference", (f_scores - l_scores).mean().round(4),
      "| band", (f_scores.std() / 5 ** 0.5).round(4))

- Forest wins 4 of 5 folds, and that is not enough to say it is better because the mean difference is about 0.002 and the band is 0.0047. The difference is smaller than the band, so it is a tie.
  By R4, I choose Logistic Regression because when scores are statistically tied, we pick the simpler, lower-latency model that requires less compute and prevents overfitting.
  (Claude fixed this line: your earlier text said 0.0014; the checkpoint value is 0.002. 0.0014 was the SelectKBest gain from Part 6.)

In [ ]:
def scan(p, y):
    rows = []
    for t in np.round(np.arange(0.20, 0.61, 0.05), 2):
        pred = (p >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
        rows.append((t, tn, fp, fn, tp,
                        round(precision_score(y, pred), 4),
                        round(recall_score(y, pred), 4),
                        round(f1_score(y, pred), 4)))
    return pd.DataFrame(rows, columns=["t", "tn", "fp", "fn", "tp", "prec", "rec", "f1"])

oof_lr = cross_val_predict(make_lr(), X_train, y_train, cv=sfk, method="predict_proba")[:, 1]
oof_rf = cross_val_predict(finalist, X_train, y_train, cv=sfk, method="predict_proba")[:, 1]
for name, o in (("LR", oof_lr), ("forest", oof_rf)):
    s = scan(o, y_train)
    print(name, "out-of-fold AUC", round(roc_auc_score(y_train, o), 4))
    print(s.to_string(index=False))
    ok = s[s["rec"] >= 0.7]
    print("chosen threshold", float(ok.loc[ok["f1"].idxmax(), "t"]))

- choose a threshold on out-of-fold predictions is important because since cross_val_predict compiles these probabilities by strictly isolating the validation rows in each fold loop, tuning your threshold on the OOF array mimics how your model will realistically perform on truly unseen data. This allows you to safely find the optimal balance between precision and recall without artificially inflating your performance scores.
- the agreement tell you about the threshold is 0.35 is the optimal, and stable threshold for a model to be rather than a statistical fluke.
- By adjusting our model threshold to 0.35, the random forest successfully detects 72.3% of potential churners (1,081 out of 1,495), allowing us to proactively save those accounts at the expected operational cost of addressing 855 false alarms.

In [ ]:
def report(name, y_true, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    print(f"{name} t={t}: tn={tn} fp={fp} fn={fn} tp={tp} | "
          f"acc={accuracy_score(y_true, pred):.4f} prec={precision_score(y_true, pred):.4f} "
          f"rec={recall_score(y_true, pred):.4f} f1={f1_score(y_true, pred):.4f}")

rf_final = finalist.fit(X_train, y_train)          # refit on ALL training rows
lr_final = make_lr().fit(X_train, y_train)
p_rf = rf_final.predict_proba(X_test)[:, 1]
p_lr = lr_final.predict_proba(X_test)[:, 1]
print("TEST AUC  forest", round(roc_auc_score(y_test, p_rf), 4),
      "| LR", round(roc_auc_score(y_test, p_lr), 4))
for t in (0.5, 0.35):
    report("forest", y_test, p_rf, t)
    report("LR    ", y_test, p_lr, t)
print("train acc forest", round(rf_final.score(X_train, y_train), 4),
      "| test acc", round(rf_final.score(X_test, y_test), 4))
old = make_forest(max_depth=5, min_samples_leaf=5).fit(X_train, y_train)
dflt = make_forest().fit(X_train, y_train)
print("Day10 forest TEST AUC", round(roc_auc_score(y_test, old.predict_proba(X_test)[:, 1]), 4),
      "| default forest", round(roc_auc_score(y_test, dflt.predict_proba(X_test)[:, 1]), 4))

## Part 8 - The final test (used once)

| Model | Test AUC | Recall at t = 0.35 |
|---|---|---|
| Baseline (always "stays") | 0.5000 (accuracy 0.7346) | 0.00 % |
| Default forest | 0.8183 | not evaluated |
| Day 10 forest (depth 5, leaf 5) | 0.8404 | not evaluated |
| Tuned forest (depth 5, leaf 20) | 0.8410 | 72.46 % |
| Logistic regression | 0.8383 | 71.93 % |

- I would ship the **logistic regression**. The decision was made in Part 7 with cross-validation (R4): the forest beat logistic regression by only about 0.002, which is smaller than the tie band (0.0047), so the two models tie and I choose the simpler, more explainable one. The test set only **confirms** it: the test AUC gap is 0.0027 (0.8410 - 0.8383), also inside the band, and the recalls at 0.35 are almost the same (72.46 % vs 71.93 %, about 2 churners out of 374).

- No. The tuned forest's test AUC is only 0.0006 above the Day 10 forest (0.8410 vs 0.8404). CV did not make the model better; it gave me two things instead: the knowledge that 12 of the 16 grid settings tie (so my Day 10 pick was never fragile), and the tie band (0.0047), which let me call a tie with evidence instead of by feeling.

- From limiting how deep the trees can grow (`max_depth` and `min_samples_leaf`), not from the search itself. The default forest scores 0.8183 on test and the tuned models about 0.840 to 0.841, a gain of about +0.023.

- The forest-minus-LR AUC gap ranged from about -0.003 to +0.008, so zero is inside the range. This means the data **cannot show** that the forest is better. The forest might be a little better (up to +0.008) or a little worse (-0.003); we do not know. "Not proven" is different from "identical", so I pick the simpler model.

- This test set has been looked at on Days 9, 10 and 12. We made no decision with it, but a stricter team would no longer treat it as sealed. They would get fresh data that nobody has touched (for example, newer customers collected after the model was built, or a second hold-out set kept locked from the start) and score the shipped model on that once.

## Part 9 - Conclusion and README

**Conclusion**

I wanted to predict which of 7,043 telecom customers will churn (26.5 % do) from tenure, charges and a few contract and service columns. I kept 20 % as a sealed test set (1,409 customers, 374 churners) and compared models only with 5-fold stratified cross-validation on the training rows, with the rules (R1 to R6) written before any search. I ran a grid of 16 settings, one-at-a-time checks and a random search, and opened the test set once at the end. In CV the default forest scored 0.8102, the tuned forests about 0.844 to 0.845 (my pick, depth 5 and leaf 20, scored 0.8442 +- 0.0105) and logistic regression 0.8422 +- 0.0122. The forest's lead is about 0.002, smaller than the tie band of 0.0047, so by R4 it is a tie and I would ship the simpler logistic regression; the test set agreed (AUC 0.8383 vs 0.8410). The real gain came from limiting tree depth and leaf size (about +0.023 over the default forest), not from the search. At the chosen threshold of 0.35 the logistic regression catches 269 of 374 churners (71.9 %), and we would contact 494 customers, 225 of whom would not have left. The limits are honest ones: the test set holds only 374 churners, so the estimate is noisy (the bootstrap gap ran from -0.003 to +0.008); the test set was reused on Days 9, 10 and 12 although no decision used it; and the features themselves cap the score at an AUC of about 0.84.